<a href="https://colab.research.google.com/github/Gilleswint-s1128612/bachelor-thesis-LambdaMART-TAS-B/blob/main/Thesis_TAS_B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
project_root = "/content/drive/MyDrive/TAS-B_MSMARCO_thesis"


In [ ]:
import os
# set Java 21
!apt-get update && apt-get install openjdk-21-jre-headless -qq > /dev/null
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
!update-alternatives --set java /usr/lib/jvm/java-21-openjdk-amd64/bin/java
!java -version

In [ ]:
!pip install pyserini
!pip install sentence_transformers
!pip install datasets
!pip install codecarbon
!pip install faiss-gpu-cu12


In [ ]:
# We reuse the bm25 docs already filtered in the LambdaMART part of the thesis
# contain pid and text
top1000_docs= "/content/drive/MyDrive/LamdaMART_MSMARCO_thesis/collections/msmarco-passage/filtered_collection.tsv"
# contain qid, pid and rank
bm25_run_log = "/content/drive/MyDrive/LamdaMART_MSMARCO_thesis/runs/run.dl19.bm25.tsv"

**TAS-B Passage Embedding**

In [ ]:
# embedding and reranking must be run after each other because the embeddings are stored in a dictionary
import os
import time
import torch
import numpy as np
import pandas as pd
from codecarbon import EmissionsTracker
from sentence_transformers import SentenceTransformer, util
from tqdm.notebook import tqdm

%cd $project_root
# create directories for runs
os.makedirs("runs", exist_ok=True)

output_run_path = "runs/run.dl19.dense.tasb.txt"

idx_tracker = EmissionsTracker(
    project_name="3_TASB_Indexing",
    output_file="emissions.csv",
    measure_power_secs=5
)

# see if GPU is available
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
# implement TAS-B
model = SentenceTransformer('sebastian-hofstaetter/distilbert-dot-tas_b-b256-msmarco', device=device)

idx_tracker.start()
# creates a dictionary of passages and their text
pid_to_text_dict = {}
# creates a dictionary of qid and pid
bm25_dict = {}

try:
    # only embed the top 1000 selected by bm25
    with open(top1000_docs, "r", encoding="utf-8", errors="ignore") as f:

      for line in f:
          parts = line.strip().split("\t", 1)
          # check if it contains the pid and text part
          if len(parts) == 2:

            pid_to_text_dict[parts[0].strip()] = parts[1].strip()

    with open(bm25_run_log, "r", encoding="utf-8") as run_file:
        for line in run_file:
            parts= line.strip().split("\t")
            # check if there is 3 args qid, pid and rank
            if len(parts) >= 2:
                qid= parts[0].strip()
                pid= parts[1].strip()
                # check if the qid is already added
                if qid not in bm25_dict:
                  bm25_dict[qid] = []
                # add the passage to the query
                bm25_dict[qid].append(pid)

    # pre index document embeddings
    print("Getting unique pids")
    unique_pids = list(pid_to_text_dict.keys())
    unique_passage_txt = [pid_to_text_dict[pid] for pid in unique_pids]

    print(f"Encoding {len(unique_pids)} passages")
    # convert every passage to embedding
    passage_embeddings_matrix = model.encode(unique_passage_txt, batch_size=256, convert_to_tensor=True, show_progress_bar=True)
    # link pid to its embedding
    passage_embedding_lookup = {pid: passage_embeddings_matrix[i] for i, pid in enumerate(unique_pids)}

finally:
    idx_tracker.stop()
    print("Indexing and embedding complete!")

**TAS-B Query embedding and dot product score**

In [ ]:

import os
import time
import torch
import numpy as np
import pandas as pd
from codecarbon import EmissionsTracker
from pyserini.search import get_topics
from sentence_transformers import SentenceTransformer, util
from tqdm.notebook import tqdm

%cd $project_root
rerank_tracker = EmissionsTracker(
    project_name="4_TASB_Reranking",
    output_file="emissions.csv",
    measure_power_secs=5
)
rerank_tracker.start()

try:
    # qid and query text
    queries_dl19 = get_topics('dl19-passage')

    #qid and pid
    qid_pid_dict = tqdm(bm25_dict.items(), desc="Neural Reranking")

    skipped_pids= 0
    total_written_lines= 0


    with open(output_run_path, "w", encoding="utf-8") as output_file:
        for qid, pids in qid_pid_dict:
            query_dict = queries_dl19.get(qid)
            if not query_dict:
                try:
                    query_dict = queries_dl19.get(int(qid))
                except ValueError:
                    pass

            if not query_dict:
                continue
            # get query text for dl19
            query_text = query_dict['title']
            # gets only the passage embeddings
            valid_embeddings = [passage_embedding_lookup[pid] for pid in pids]

            if not valid_embeddings:
                continue
            # stack embeddings together in their original order, to remain index positions for pids
            current_passage_matrix = torch.stack(valid_embeddings)
            # embed the queries
            query_embedding = model.encode(query_text, convert_to_tensor=True, show_progress_bar=True)
            # compute dot product between query embeddings and passage
            scores = util.dot_score(query_embedding, current_passage_matrix)[0]
            # rank scores, contains rank and index linked to passage
            ranked_indices = torch.argsort(scores, descending=True).cpu().numpy()
            # write the results in the follwoing way Qid, Pid, Rank
            for rank_idx, local_idx in enumerate(ranked_indices, start=1):
                output_file.write(f"{qid}\t{pids[local_idx]}\t{rank_idx}\n")
                total_written_lines += 1
    print(f"Total lines written to run file: {total_written_lines}")

    print(f"Total pids skipped{skipped_pids}")

except Exception as e:
    print(f"ERROR: {e}")
finally:
    rerank_tracker.stop()
    print("TAS-B reranking complete")

**TAS-B results**

In [ ]:
%cd $project_root
!python3 -m pyserini.eval.convert_msmarco_run_to_trec_run \
  --input runs/run.dl19.dense.tasb.txt \
  --output runs/run.dl19.dense.tasb.trec

!python3 -m pyserini.eval.trec_eval -c -m ndcg_cut.10 dl19-passage runs/run.dl19.dense.tasb.trec

In [ ]:
%cd $project_root
import pandas as pd

try:
    df = pd.read_csv("emissions.csv")
    columns_to_show = ["project_name", "duration", "emissions", "cpu_power", "gpu_power", "ram_power", "energy_consumed"]
    available_columns = [col for col in columns_to_show if col in df.columns]
    display(df[available_columns])

except FileNotFoundError:
    print("Emissions.csv not found.")
    df = None